# Rover on Rough Terrain — Planning Challenge (C++ edition, 30 min)

A point rover must drive from `start` to `goal` across rough 2D terrain.

Everything is bundled in a `World` value called **`w`** (loaded once in Setup and
passed to every function). **Convention:** points are `(row, col)`, so a cost map
is indexed `cost_map[row][col]`.

**`w` gives you:**
- `w.features` — a `(H × W × 3)` tensor (`std::vector<std::vector<std::array<float,3>>>`);
  `w.features[r][c]` is `[rock, mud, slope]` at that cell (rock is 0/1 and lethal;
  mud, slope ∈ 0..1).
- `w.example_cost_map` — a `(H × W)` **example** cost grid
  (`std::vector<std::vector<double>>`, uniform 1.0, rocks `= ROCK_PENALTY`), *not* the
  true cost. It ignores mud/slope, so a path planned under it costs too much → it's
  the starting point you improve on.
- `w.demos` — a few expert `(row, col)` polylines, each between its *own* start/goal
  (not yours). No single one is the answer; they reveal which terrain to avoid.
- `w.start`, `w.goal` — `(row, col)` endpoints; `w.H`, `w.W` — grid size.
- `astar_plan(w, cost_map)` — a **provided** planner (`planner_p1.hpp`). Give it a
  cost map, get a near-optimal `(row, col)` path. You do **not** write a planner.

**Your one task:** write `cost_map_learned(w)` (in `learn_p1.hpp`) → a cost map that,
routed through `astar_plan`, yields a path as terrain-cheap as the experts'.
You only `%%writefile learn_p1.hpp`.

### Setup

In [ ]:
import os, subprocess, struct
import numpy as np
if not os.path.exists('field_p1.npy'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field_p1.npy').astype(np.float32)        # (H, W, 3)
demos = np.load('demos_p1.npy', allow_pickle=True)        # object array of (T,2)
meta  = np.load('meta_p1.npz')
H, W = field.shape[:2]

with open('world_p1.bin', 'wb') as f:
    f.write(struct.pack('<ii', H, W))
    f.write(np.ascontiguousarray(field, np.float32).tobytes())
    f.write(np.ascontiguousarray(meta['true_cost'], np.float64).tobytes())
    f.write(np.asarray(meta['start'], np.float64).tobytes())
    f.write(np.asarray(meta['goal'],  np.float64).tobytes())
    f.write(struct.pack('<ddd', float(meta['optimal_cost']), float(meta['tol']),
                        float(meta['rock_sentinel'])))
    f.write(struct.pack('<i', len(demos)))
    for d in demos:
        d = np.asarray(d, np.float64)
        f.write(struct.pack('<i', d.shape[0]))
        f.write(np.ascontiguousarray(d, np.float64).tobytes())

import sys
sys.path.insert(0, os.getcwd())
from viz_p1 import show_cpp, build_and_run

print('g++ version:',
      subprocess.run(['g++', '--version'], capture_output=True, text=True).stdout.splitlines()[0])
print(f'packed world_p1.bin: field {field.shape} | demos {len(demos)} '
      f'| start {meta["start"]} goal {meta["goal"]}')
# Compile-check the shipped baseline driver (needs no candidate code), so a bad
# clone or toolchain surfaces here rather than inside the TODO.
_chk = subprocess.run(['g++', '-O2', '-std=c++17', 'main_run_baseline_p1.cpp', '-o', '_chk_p1'],
                      capture_output=True, text=True)
print('toolchain + shipped sources:', _chk.stderr.strip() or 'OK')

### Baseline run &mdash; expect `FAIL`

In [ ]:
build_and_run('main_run_baseline_p1.cpp', [])

## TODO &mdash; `cost_map_learned` in `learn_p1.hpp` (see the cell)

In [ ]:
%%writefile learn_p1.hpp
// ---- TODO: learn a (H x W) cost map from the demos -----------------------
// Make terrain the experts drive through cheap, and what they detour around
// costly, so astar_plan routes like them. You only need the RANKING right, not
// the true cost numerically. Keep rocks at ROCK_PENALTY.
// (e.g. count w.demos visitation, or weight up mud/slope -- your call.)
//
// API (all in w; points are (row,col); do not edit the plumbing):
//   w.features[r][c]      : {rock, mud, slope} at cell (r,c) (float[3])
//   w.example_cost_map    : (H x W) example Grid (uniform; rocks = ROCK_PENALTY)
//   w.demos               : vector<Path>, expert (row,col) polylines (q.r, q.c)
//   w.start, w.goal       : Vec2{r, c}; w.H, w.W : grid size
//   ROCK, MUD, SLOPE      : channel indices 0,1,2; ROCK_PENALTY : untraversable
//   Grid = vector<vector<double>> (index [row][col]); Path = vector<Vec2>
//   astar_plan(w, cm)     : -> Path, near-optimal (row,col) path under Grid cm
//   path_cost(w, p, cm)   : -> double; evaluate(w, p, cm) : -> bool (prints)
#pragma once
#include "metric_p1.hpp"
#include "terrain_p1.hpp"

inline Grid cost_map_learned(const World& w) {
    // TODO: use w.demos. (Falls back to the w.example_cost_map for now.)
    return w.example_cost_map;
}

### Run with your learned cost &mdash; goal: `PASS`

In [ ]:
build_and_run('main_run_p1.cpp', [('run_cost_p1.bin', 'learned: planned path over your learned cost map'), ('run_true_p1.bin', 'learned: planned path over the true cost map')])

### Discussion (optional — no code needed)

Be ready to talk through: where your learned cost generalizes badly; A* vs.
sampling vs. trajectory-opt and what each guarantees; the planner's Euclidean
heuristic (admissible here?); 8- vs 4-connectivity and resolution tradeoffs;
`float` vs `double` / cache locality in C++; a safety margin around rocks;
replanning in a 50 Hz loop.